In [1]:
from rag_helper import RAGBase
from openai import OpenAI

In [4]:
import os

In [5]:
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

### Asking without tools

In [6]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
)

response.output_text

"You'd like to join a course, however, I need a bit more information. Could you please tell me what course you're referring to and when it started? That way, I can try to help you figure out if it's still possible to join."

### Defining the tool

In [ ]:
from ingest import load_faq_data, build_index

In [ ]:
documents = load_faq_data()
print(f"Loaded {len(documents)} documents")
index = build_index(documents)

In [ ]:
def search(query):
   boost_dict = {"question": 2.0, "section": 0.5}
   filter_dict = {"course": "llm-zoomcamp"}

   return index.search(
     query, 
     boost_dict=boost_dict,
     filter_dict=filter_dict,
     num_results=5
  )

In [ ]:
search_results = search("How do I run Ollama?")
search_results

In [ ]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

### Sending the question with the tool

In [ ]:
import json

# 1) Check messages serializability
try:
    json_str = json.dumps(messages)
    print("messages OK (len bytes):", len(json_str))
except Exception as e:
    print("messages not serializable:", type(e).__name__, e)

# 2) Check tools/search_tool serializability
try:
    json_str = json.dumps([search_tool])
    print("tools OK (len bytes):", len(json_str))
except Exception as e:
    print("tools not serializable:", type(e).__name__, e)

# 3) Inspect contents briefly
print("messages preview:", messages[:2])
print("search_tool preview keys:", list(search_tool.keys()))

In [ ]:
response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
    tools=[search_tool],
)

response.output

### Executing the function and sending the result back

In [ ]:
import json
call = response.output[1]
call
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)


In [ ]:
import json
# Append the assistant's text output as a proper assistant message
messages.append({
    "role": "assistant",
    "content": response.output_text
})
# Append the tool result as a tool message so the model can consume it
messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})
messages

In [ ]:
import json
try:
    json.dumps(messages)
    print("messages OK")
except TypeError as e:
    print("NOT serializable:", e)

In [ ]:
response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    tools=[search_tool],
)

response.output_text

In [ ]:
response.output

### The Agentic Loop

#### A developer prompt

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

### A function-call helper

In [ ]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

### Processing one response

In [ ]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    tools=[search_tool],
)
messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

### The full agent loop

In [ ]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="llama-3.3-70b-versatile",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        print("item:", type)
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

### Wrapping it in a function

In [ ]:
def agent_loop(instructions, question, model="llama-3.3-70b-versatile") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

In [ ]:
agent_loop(instructions, "How do I run Olama locally?")

In [ ]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

### Encouraging multiple searches

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [ ]:
agent_loop(instructions, "I just discovered the course. Can I join it?")

### Restricting off-topic questions

In [ ]:
agent_loop(instructions, "what's queen gambit?")

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [ ]:
agent_loop(instructions, "what's queen gambit?")